# SHIPP — 92 Release Validation\n\n**Purpose:** final read-only validation after Tasks 90 and 91.\n\nThis task does **not** rebuild Silver, Gold, RAG, Requests, or Categories. It validates the existing canonical products and appends evidence to `bootcamp_students.shipp_gold.release_validation_log`.\n\nRequests and their categories are already persisted by the canonical `silver_requests` pipeline. This gate verifies that they are preserved and that Gold keeps the same request/category contract.\n\nNo passwords, tokens, user credentials, or Databricks secrets are written to the log.

In [0]:
%run ../common/shipp_silver_lib

In [0]:
import importlib\nimport json\nimport uuid\nfrom datetime import datetime, timezone\n\nrelease_validation = importlib.reload(\n    importlib.import_module("data_pipeline.quality.release_validation")\n)\n\nTABLE_MAP = {\n    "silver_listings": SILVER_LISTINGS,\n    "silver_requests": SILVER_REQUESTS,\n    "silver_routes": SILVER_ROUTES,\n    "gold_candidate_matches": GOLD_CANDIDATE_MATCHES,\n    "gold_listing_search_docs": GOLD_LISTING_SEARCH_DOCS,\n    "gold_marketplace_metrics": GOLD_MARKETPLACE_METRICS,\n    "rag_eval_results": RAG_EVAL_RESULTS,\n    "pipeline_run_log": PIPELINE_RUN_LOG,\n}\n\nVALIDATION_LOG = f"{CATALOG}.{GOLD_SCHEMA}.release_validation_log"\nVALIDATION_RUN_ID = str(uuid.uuid4())\n\nprint("VALIDATION_RUN_ID =", VALIDATION_RUN_ID)\nprint("VALIDATION_LOG    =", VALIDATION_LOG)

In [0]:
summary, checks = release_validation.run_release_validation(spark, TABLE_MAP)\n\ncheck_rows = [\n    {\n        "check_name": c.check_name,\n        "status": c.status,\n        "observed_value": c.observed_value,\n        "detail": c.detail,\n    }\n    for c in checks\n]\n\nprint(json.dumps(summary, indent=2, default=str))\ndisplay(spark.createDataFrame(check_rows).orderBy("status", "check_name"))

In [0]:
# Persist validation evidence only. Existing business products are untouched.\nspark.sql(f"""\nCREATE TABLE IF NOT EXISTS {VALIDATION_LOG} (\n    validation_run_id STRING,\n    checked_at TIMESTAMP,\n    overall_status STRING,\n    check_name STRING,\n    check_status STRING,\n    observed_value STRING,\n    detail STRING,\n    request_count BIGINT,\n    request_categories STRING,\n    gold_match_count BIGINT,\n    search_doc_count BIGINT\n) USING DELTA\n""")\n\nchecked_at = datetime.now(timezone.utc).replace(tzinfo=None)\nrequest_categories_json = json.dumps(summary.get("request_categories", []))\n\nrows = [\n    (\n        VALIDATION_RUN_ID,\n        checked_at,\n        summary["status"],\n        c.check_name,\n        c.status,\n        c.observed_value,\n        c.detail,\n        int(summary.get("request_count", 0)),\n        request_categories_json,\n        int(summary.get("gold_match_count", 0)),\n        int(summary.get("search_doc_count", 0)),\n    )\n    for c in checks\n]\n\nLOG_SCHEMA = (\n    "validation_run_id string, checked_at timestamp, overall_status string, "\n    "check_name string, check_status string, observed_value string, detail string, "\n    "request_count long, request_categories string, gold_match_count long, search_doc_count long"\n)\n\nspark.createDataFrame(rows, LOG_SCHEMA) \\\n    .write.format("delta") \\\n    .mode("append") \\\n    .saveAsTable(VALIDATION_LOG)\n\ndisplay(\n    spark.table(VALIDATION_LOG)\n    .filter(F.col("validation_run_id") == VALIDATION_RUN_ID)\n    .orderBy("check_status", "check_name")\n)\n\nprint("Saved request count      :", summary.get("request_count"))\nprint("Saved request categories :", summary.get("request_categories"))

In [0]:
result = {\n    "validation_run_id": VALIDATION_RUN_ID,\n    "log_table": VALIDATION_LOG,\n    **summary,\n}\n\nif summary["status"] != "PASS":\n    raise AssertionError(\n        "RELEASE VALIDATION FAILED: " + ", ".join(summary["failed_checks"])\n    )\n\nprint("PASS — SHIPP release validation completed.")\ndbutils.notebook.exit(json.dumps(result, default=str))